# Exporting and checking quantization for the KWS model

This notebook loads the trained PyTorch model, converts it to a `.tflite` model with LiteRT Torch, verifies that the exported model produces similar outputs on a sample input, **and then inspects the exported `.tflite` file to determine whether it is actually quantized**.

Important note for ESP32-WROOM:
- A plain `.tflite` export is **not automatically the same thing as a fully int8-quantized model**.
- For embedded deployment you usually want:
  - `int8` weights
  - `int8` activations
  - ideally `int8` input/output too
- This notebook makes that status explicit after export.


In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from model import DSCNN

try:
    import litert_torch
except ImportError:
    print("Error: litert_torch is not installed. Run: pip install litert-torch", file=sys.stderr)
    raise


def try_get_tflite_interpreter():
    try:
        from tensorflow.lite import Interpreter
        return Interpreter, "tensorflow.lite.Interpreter"
    except Exception:
        pass

    try:
        from tflite_runtime.interpreter import Interpreter
        return Interpreter, "tflite_runtime.Interpreter"
    except Exception:
        pass

    return None, None


I0000 00:00:1773854379.598183  748193 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773854379.621855  748193 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773854381.734830  748193 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.16.

## Configuration

Set the checkpoint path and the input shape your model expects.

- For your original setup, use `N_MELS=64` and `TIME_FRAMES=101`.
- For the smaller embedded-friendly setup, change these to `40` and `49` if that is what the model was trained on.

Quantization settings:
- `CHECK_TFLITE_WITH_INTERPRETER=True` tries to inspect the exported model using either `tensorflow.lite.Interpreter` or `tflite_runtime.Interpreter`.
- `REQUIRE_INT8_IO=True` makes the notebook clearly fail the quantization check unless both model input and model output are int8.


In [2]:
CHECKPOINT_PATH = Path("model_files/kws_model.pt")
EXPORT_DIR = Path("model_files")
OUTPUT_TFLITE = EXPORT_DIR / "kws_model.tflite"
OUTPUT_CC = EXPORT_DIR / "model_data.cc"
OUTPUT_H = EXPORT_DIR / "model_data.h"

NUM_CLASSES = 4
N_MELS = 64
TIME_FRAMES = 101

# Keep this as batch=1 for export / deployment shape.
SAMPLE_INPUT_SHAPE = (1, 1, N_MELS, TIME_FRAMES)

# Inspection options
CHECK_TFLITE_WITH_INTERPRETER = True
REQUIRE_INT8_IO = True

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Checkpoint:", CHECKPOINT_PATH)
print("TFLite output:", OUTPUT_TFLITE)
print("Sample input shape:", SAMPLE_INPUT_SHAPE)


Checkpoint: model_files/kws_model.pt
TFLite output: model_files/kws_model.tflite
Sample input shape: (1, 1, 64, 101)


## Load the pre-trained model


In [3]:
def load_checkpoint(model: nn.Module, checkpoint_path: Path) -> nn.Module:
    ckpt = torch.load(checkpoint_path, map_location="cpu")

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state_dict = ckpt["model_state_dict"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt

    cleaned = {}
    for k, v in state_dict.items():
        cleaned[k[7:] if k.startswith("module.") else k] = v

    model.load_state_dict(cleaned, strict=True)
    model.eval()
    return model


model = DSCNN(num_classes=NUM_CLASSES)
model = load_checkpoint(model, CHECKPOINT_PATH)

print("Model loaded successfully.")
print(model)


Model loaded successfully.
DSCNN(
  (model): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Sequential(
      (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU()
    )
    (4): Sequential(
      (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)


## Create a representative example input

The converter needs a sample input with the exact shape the model will see at inference time.


In [4]:
sample_input = torch.randn(*SAMPLE_INPUT_SHAPE, dtype=torch.float32)

with torch.no_grad():
    torch_output = model(sample_input).detach().cpu().numpy()

print("Sample input shape:", tuple(sample_input.shape))
print("PyTorch output shape:", torch_output.shape)
print("PyTorch output:", torch_output)


Sample input shape: (1, 1, 64, 101)
PyTorch output shape: (1, 4)
PyTorch output: [[ -3.3422697   8.79804   -12.559123   -3.5315936]]


## Convert the model to TFLite with LiteRT Torch


In [5]:
edge_model = litert_torch.convert(model, (sample_input,))
edge_model.export(str(OUTPUT_TFLITE))

print(f"Saved TFLite model to: {OUTPUT_TFLITE}")


INFO:tensorflow:Assets written to: /tmp/tmpln1trxlw/assets


INFO:tensorflow:Assets written to: /tmp/tmpln1trxlw/assets


Saved TFLite model to: model_files/kws_model.tflite


W0000 00:00:1773854387.686549  748193 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773854387.686587  748193 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1773854387.686912  748193 reader.cc:83] Reading SavedModel from: /tmp/tmpln1trxlw
I0000 00:00:1773854387.687519  748193 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1773854387.687525  748193 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpln1trxlw
I0000 00:00:1773854387.695335  748193 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1773854387.696386  748193 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1773854387.740073  748193 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpln1trxlw
I0000 00:00:1773854387.750382  748193 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 63477 microseconds.
I0000 00:00:1773854387.763501  748193

## Verify the exported model numerically


In [6]:
edge_output = edge_model(sample_input)

print("LiteRT output shape:", np.array(edge_output).shape)
print("LiteRT output:", edge_output)

is_close = np.allclose(torch_output, edge_output, atol=1e-4, rtol=1e-4)
max_abs_diff = np.max(np.abs(torch_output - edge_output))

print("Outputs close:", is_close)
print("Max absolute difference:", max_abs_diff)


LiteRT output shape: (1, 4)
LiteRT output: [[ -3.3422656   8.798044  -12.559133   -3.5315986]]
Outputs close: True
Max absolute difference: 9.536743e-06


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
ERROR: Subgraph reshaping is enabled by default, TFLITE_XNNPACK_DELEGATE_FLAG_ENABLE_SUBGRAPH_RESHAPING is deprecated and will be removed in the future.


## Inspect whether the exported `.tflite` model is really quantized

This section checks the exported model with a TFLite interpreter if one is available.

What we want for ESP32-WROOM:
- input dtype: `int8`
- output dtype: `int8`
- non-trivial quantization parameters (scale / zero-point)

If neither TensorFlow Lite nor `tflite_runtime` is installed, this section will explain that the export succeeded but **full int8 quantization is still unverified**.


In [7]:
from ai_edge_litert.interpreter import Interpreter

Interpreter, interpreter_name = try_get_tflite_interpreter()

def inspect_tflite_quantization(tflite_path: Path):
    if Interpreter is None:
        print("No TFLite interpreter is installed in this environment.")
        print("Install either TensorFlow or tflite_runtime to inspect quantization status.")
        return {
            "checked": False,
            "reason": "no_interpreter",
        }

    interpreter = Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print(f"Inspector backend: {interpreter_name}")
    print("\nInput details:")
    for i, d in enumerate(input_details):
        print(f"  input[{i}] name={d.get('name')} dtype={d.get('dtype')} shape={d.get('shape')}")
        print(f"           quantization={d.get('quantization')} quantization_parameters={d.get('quantization_parameters')}")

    print("\nOutput details:")
    for i, d in enumerate(output_details):
        print(f"  output[{i}] name={d.get('name')} dtype={d.get('dtype')} shape={d.get('shape')}")
        print(f"            quantization={d.get('quantization')} quantization_parameters={d.get('quantization_parameters')}")

    in_dtypes = [d.get("dtype") for d in input_details]
    out_dtypes = [d.get("dtype") for d in output_details]

    all_in_int8 = all(dtype == np.int8 for dtype in in_dtypes)
    all_out_int8 = all(dtype == np.int8 for dtype in out_dtypes)

    def has_real_quant_params(details):
        ok = True
        for d in details:
            scale, zp = d.get("quantization", (0.0, 0))
            if scale in (0.0, None):
                ok = False
        return ok

    in_has_quant = has_real_quant_params(input_details)
    out_has_quant = has_real_quant_params(output_details)

    fully_int8 = all_in_int8 and all_out_int8 and in_has_quant and out_has_quant

    print("\nQuantization summary:")
    print("  all inputs int8:", all_in_int8)
    print("  all outputs int8:", all_out_int8)
    print("  inputs have quant params:", in_has_quant)
    print("  outputs have quant params:", out_has_quant)
    print("  fully int8 model interface:", fully_int8)

    return {
        "checked": True,
        "backend": interpreter_name,
        "all_inputs_int8": all_in_int8,
        "all_outputs_int8": all_out_int8,
        "inputs_have_quant_params": in_has_quant,
        "outputs_have_quant_params": out_has_quant,
        "fully_int8_interface": fully_int8,
    }

quant_report = inspect_tflite_quantization(OUTPUT_TFLITE)

if REQUIRE_INT8_IO:
    if not quant_report.get("checked", False):
        print("\nWARNING: Quantization could not be verified in this environment.")
        print("         The model may still be float or hybrid.")
    elif not quant_report.get("fully_int8_interface", False):
        raise RuntimeError(
            "Exported TFLite model is not fully int8 at the interface. "
            "This is likely still too large or unsuitable for ESP32-WROOM."
        )
    else:
        print("\nPASS: Exported model has int8 input/output interface.")


No TFLite interpreter is installed in this environment.
Install either TensorFlow or tflite_runtime to inspect quantization status.

         The model may still be float or hybrid.


## What to do if the model is still float or hybrid

If the inspection above shows float32 input/output, then the notebook successfully exported a `.tflite` model, but it did **not** give you the fully quantized embedded model you want.

That is an important result, because for your ESP32-WROOM target:

- float32 is usually too large
- hybrid quantization may still be too large
- you really want a fully int8 deployment path

At that point, your next step is **not** to trust the file blindly. Treat it as an export artifact for testing, not yet as the final embedded model.


## Optional: generate a C array for ESP-IDF / TFLite Micro

This writes `model_data.cc` and `model_data.h` with an aligned `g_model` array.


In [8]:
def write_tflite_as_c_array(
    input_file: Path,
    output_cc: Path,
    output_h: Path,
    array_name: str = "g_model",
    bytes_per_line: int = 12,
) -> None:
    data = input_file.read_bytes()

    with open(output_h, "w", encoding="utf-8") as f:
        f.write("#ifndef MODEL_DATA_H_\n")
        f.write("#define MODEL_DATA_H_\n\n")
        f.write("#include <cstdint>\n\n")
        f.write(f"extern const unsigned char {array_name}[];\n")
        f.write(f"extern const int {array_name}_len;\n\n")
        f.write("#endif  // MODEL_DATA_H_\n")

    with open(output_cc, "w", encoding="utf-8") as f:
        f.write('#include "model_data.h"\n\n')
        f.write(f"alignas(16) const unsigned char {array_name}[] = {{\n")

        for i, b in enumerate(data):
            if i % bytes_per_line == 0:
                f.write("    ")
            f.write(f"0x{b:02x}, ")
            if i % bytes_per_line == bytes_per_line - 1:
                f.write("\n")

        if len(data) % bytes_per_line != 0:
            f.write("\n")

        f.write("};\n\n")
        f.write(f"const int {array_name}_len = {len(data)};\n")


write_tflite_as_c_array(OUTPUT_TFLITE, OUTPUT_CC, OUTPUT_H)

print(f"Saved C array source to: {OUTPUT_CC}")
print(f"Saved C array header to: {OUTPUT_H}")


Saved C array source to: model_files/model_data.cc
Saved C array header to: model_files/model_data.h


## Next steps

- If the quantization inspection passes with int8 input/output, copy `model_data.cc` and `model_data.h` into your ESP-IDF component.
- If the model is still float or hybrid, do **not** assume it will fit on ESP32-WROOM.
- Use `g_model` and `g_model_len` in your TFLite Micro test app.
- If the model is too large for ESP32-WROOM, reduce the feature size and/or the model size before exporting again.
- For your project, a smaller feature size such as `40 x 49` and a smaller DSCNN is much more realistic than `64 x 101`.
